# Aquisição de dados públicos e construção dos bancos cisTarget customizados

Complemento executável da Seção 3.2 da metodologia da dissertação. Reúne a aquisição dos dados
públicos, a definição dos seis grupos de fatores de transcrição avaliados e a construção dos
bancos cisTarget usados nas duas condições do trabalho: apenas motivos, e motivos mais ChIP-seq.

| Seção | Etapa |
|---|---|
| 0 | Configuração, diretórios e registro do ambiente |
| 1 | Aquisição dos dados públicos |
| 2 | Seleção do melhor experimento de ChIP-seq por fator no ENCODE |
| 3 | Definição dos seis grupos de fatores de transcrição |
| 4 | Construção dos bancos cisTarget customizados |
| 5 | Relatório final |

Fora as definições da Seção 0, nenhuma seção usa variáveis criadas por outra: o que passa de
uma etapa para a seguinte é gravado em disco e lido de lá. Assim, qualquer seção pode ser
executada sozinha depois da Seção 0, desde que as anteriores já tenham rodado alguma vez.
Arquivos que já existem não são refeitos, a menos que `FORCAR = True`, o que refaz tudo,
inclusive os downloads.

Requisitos: Python 3.8 ou mais recente, `pandas`, `pyarrow` e `openpyxl`. Na Seção 4, cada
fator ocupa cerca de 1,4 GB em disco (banco combinado mais `.tbl`), e são necessários 16 GB de
RAM.

## Estrutura de diretórios

```
projeto/
├── analise/                        este notebook e a tabela curada do ENCODE
├── downloads/                      dados de terceiros
│   ├── geo/GSM5677000/             matriz de expressão scRNA-seq
│   ├── encode/                     metadados do ENCODE
│   └── literatura/                 tabelas suplementares
├── data/                           arquivos consumidos pelo pySCENIC
│   ├── tfs/                        allTFs_hg38.txt
│   ├── feathers/
│   │   ├── motifs/                 banco de referência
│   │   ├── all_tracks_ofc/         feather global de tracks
│   │   ├── <grupo>/only_chip/      feather recortado, um por fator
│   │   └── <grupo>/motifs_plus_chip/
│   └── tbls/
│       ├── motifs/                 anotação de referência
│       └── <grupo>/                anotação customizada, uma por fator
└── outs/                           listas, checklists e relatórios
```

`<grupo>` é a pasta de cada grupo, definida na Seção 0.3.

# 0. Configuração

## 0.1. Imports e raiz do projeto

A raiz do projeto vem da variável de ambiente `PROJETO_RAIZ`. Sem ela, o notebook sobe a partir
do diretório de trabalho até achar uma pasta que contenha pelo menos duas entre `analise/`,
`data/`, `downloads/` e `outs/`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import re
import sys
import urllib.request
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Callable

import pandas as pd
import pyarrow.feather as pf

FORCAR = False


def encontrar_raiz(inicio: Path | None = None) -> Path:
    """Resolve a raiz por PROJETO_RAIZ, por estar dentro de `analise/`, ou pelos
    diretórios conhecidos acima do diretório de trabalho."""
    if os.environ.get("PROJETO_RAIZ"):
        return Path(os.environ["PROJETO_RAIZ"]).expanduser().resolve()

    inicio = (inicio or Path.cwd()).resolve()
    if inicio.name == "analise":
        return inicio.parent

    marcadores = ("data", "downloads", "outs", "analise")
    for candidato in (inicio, *inicio.parents):
        if sum((candidato / m).is_dir() for m in marcadores) >= 2:
            return candidato
    return inicio


ROOT = encontrar_raiz()
print("Raiz do projeto:", ROOT)

## 0.2. Caminhos e parâmetros

Arquivos de entrada e arquivos intermediários que passam de uma seção para outra. Os caminhos
dos arquivos gerados para cada fator na Seção 4 são montados por `caminhos_do_fator`, na 0.3.

In [ ]:
ANALISE = ROOT / "analise"
DATA = ROOT / "data"
DOWNLOADS = ROOT / "downloads"
OUTS = ROOT / "outs"

FEATHERS = DATA / "feathers"
TBLS = DATA / "tbls"

# Bancos cisTarget de referência, condição "apenas motivos"
FEATHER_MOTIFS = (
    FEATHERS / "motifs"
    / "hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather"
)
TBL_MOTIFS = TBLS / "motifs" / "motifs-v9-nr.hgnc-m0.001-o0.0.tbl"
ALLTFS = DATA / "tfs" / "allTFs_hg38.txt"

# Feather global com todos os tracks de ChIP-seq
FEATHER_TRACKS_GLOBAL = (
    FEATHERS / "all_tracks_ofc" / "pyscenic_1000_alltracks.genes_vs_tracks.rankings.feather"
)

ENCODE_METADATA_TSV = DOWNLOADS / "encode" / "metadata.tsv"
ENCODE_BEST_TSV = ANALISE / "best_bigwig_per_tf3.tsv"

GSM_SCRNA = "GSM5677000"
DIR_GEO = DOWNLOADS / "geo" / GSM_SCRNA

# Tabelas suplementares baixadas à mão das publicações: fatores expressos em HepG2 e
# ranking de enrichment score de Peng et al., de onde saem os canônicos preditos
DIR_LITERATURA = DOWNLOADS / "literatura"
ARQ_PENG_FIG2B = DIR_LITERATURA / "elife-88936-fig2-data1-v1.xlsx"
ARQ_TFS_EXPRESSOS = DIR_LITERATURA / "TFs_expressos_hepg2.csv"

# Arquivos que ligam uma seção à seguinte
ARQ_CHECKLIST = OUTS / "checklist_dados_entrada.tsv"
ARQ_AMBIENTE = OUTS / "ambiente_aquisicao_dados.json"
ARQ_GRUPOS = OUTS / "grupos_tfs.csv"
ARQ_GRUPOS_RESUMO = OUTS / "grupos_tfs_resumo.csv"
ARQ_GRUPOS_SOBREPOSICOES = OUTS / "grupos_tfs_sobreposicoes.csv"
ARQ_CANONICOS_RANKING = OUTS / "canonicos_preditos_ranking.csv"
ARQ_MAPA_TRACKS = OUTS / "mapa_tf_tracks.tsv"
ARQ_FATORES = OUTS / "fatores_a_processar.csv"
ARQ_GERACAO = OUTS / "geracao_bancos_cistarget.csv"
ARQ_CONFERENCIA = OUTS / "conferencia_bancos_cistarget.csv"
ARQ_INVENTARIO = OUTS / "inventario_arquivos_gerados.tsv"

DIRETORIOS_BASE = [
    ANALISE, DATA, DOWNLOADS, OUTS,
    DATA / "tfs", FEATHERS / "motifs", FEATHERS / "all_tracks_ofc",
    TBLS / "motifs", DIR_GEO, DOWNLOADS / "encode", DIR_LITERATURA,
]

## 0.3. Os seis grupos de fatores de transcrição

Para cada grupo ficam definidos a chave usada no código, o rótulo das impressões, a pasta dos
arquivos, o prefixo dos nomes de arquivo e um resumo do critério de seleção, que é gravado junto
com a composição dos grupos em `outs/grupos_tfs.csv`.

Com pasta e prefixo, o caminho já indica por qual critério o fator entrou:
`data/tbls/pioneiros_literatura/ptf_FOXA1.tbl` é o `.tbl` de FOXA1 como pioneiro de
literatura. Quando um fator está em dois grupos, como FOXA2, que é central de Odom e pioneiro
de literatura, cada grupo tem os seus próprios arquivos e um não sobrescreve o outro.

Os três caminhos de um fator são sempre montados por `caminhos_do_fator`, então reorganizar as
pastas exige mudar só esta célula.

In [ ]:
@dataclass(frozen=True)
class Grupo:
    chave: str
    rotulo: str
    pasta: str
    prefixo: str
    criterio: str


GRUPOS: dict[str, Grupo] = {
    g.chave: g for g in [
        Grupo("centrais_literatura", "Centrais (literatura)",
              "centrais_literatura", "ctf",
              "lista primária de Odom et al. ∩ ENCODE"),
        Grupo("centrais_se", "Centrais (super-enhancer)",
              "centrais_se", "setf",
              "lista primária de Saint-André et al. ∩ ENCODE"),
        Grupo("pioneiros_literatura", "Pioneiros (literatura)",
              "pioneiros_literatura", "ptf",
              "pioneiros descritos ∩ expressos em HepG2 ∩ ENCODE, "
              "menos os sem evidência (Seção 3.4)"),
        Grupo("canonicos_literatura", "Canônicos (literatura)",
              "canonicos_literatura", "cntf",
              "ensaio direto de não ligação a nucleossomo ∩ expressos ∩ ENCODE"),
        Grupo("pioneiros_preditos", "Pioneiros (preditos)",
              "pioneiros_preditos", "pptf",
              "Tabela 4 de Peng et al. menos pioneiros conhecidos ∩ ENCODE"),
        Grupo("canonicos_preditos", "Canônicos (preditos)",
              "canonicos", "canonico",
              "quartil inferior do enrichment score de Peng et al. ∩ expressos ∩ ENCODE, "
              "menos os já classificados"),
    ]
}


def caminhos_do_fator(chave_grupo: str, tf: str) -> dict[str, Path]:
    g = GRUPOS[chave_grupo]
    tf = tf.upper()
    return {
        "only_chip": FEATHERS / g.pasta / "only_chip" / f"{g.prefixo}_{tf}.feather",
        "motifs_plus_chip": (
            FEATHERS / g.pasta / "motifs_plus_chip"
            / f"hg38__motifs_plus_{tf}.genes_vs_motifs.rankings.feather"
        ),
        "tbl": TBLS / g.pasta / f"{g.prefixo}_{tf}.tbl",
    }


for grupo in GRUPOS.values():
    DIRETORIOS_BASE += [
        FEATHERS / grupo.pasta / "only_chip",
        FEATHERS / grupo.pasta / "motifs_plus_chip",
        TBLS / grupo.pasta,
    ]

for grupo in GRUPOS.values():
    print(f"{grupo.chave:<24} {grupo.rotulo}")

## 0.4. Criar os diretórios

In [ ]:
for d in DIRETORIOS_BASE:
    d.mkdir(parents=True, exist_ok=True)

print(f"{len(DIRETORIOS_BASE)} diretórios verificados ou criados sob {ROOT}")

## 0.5. Funções utilitárias

Download, registro no checklist de entrada e funções de apoio usadas nas seções seguintes.

`carregar_ou_gerar` recebe o caminho de um resultado, a função que o gera e a função que o lê.
Se o arquivo ainda não existe, ele é gerado; nos dois casos, o valor devolvido é lido do disco.

In [ ]:
USER_AGENT = "Mozilla/5.0 (compatible; aquisicao_dados.ipynb)"
COLUNAS_CHECKLIST = ["arquivo", "status", "bytes", "sha256", "url", "fonte"]


def sha256(caminho: Path, bloco: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for pedaco in iter(lambda: f.read(bloco), b""):
            h.update(pedaco)
    return h.hexdigest()


def rel(caminho: Path) -> str:
    try:
        return str(Path(caminho).resolve().relative_to(ROOT))
    except ValueError:
        return str(caminho)


def carregar_ou_gerar(caminho: Path, gerar: Callable[[], None], ler: Callable[[], object],
                      forcar: bool = False):
    """Gera o arquivo apenas se ele não existir, e devolve sempre a leitura do disco."""
    caminho = Path(caminho)
    if caminho.exists() and not (forcar or FORCAR):
        print(f"[disco]   {rel(caminho)}")
    else:
        gerar()
        if not caminho.exists():
            raise FileNotFoundError(f"A geração não produziu {rel(caminho)}.")
        print(f"[gerado]  {rel(caminho)}")
    return ler()


def baixar(url: str, destino: Path, forcar: bool = False, timeout: int = 300) -> Path:
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.exists() and not (forcar or FORCAR):
        print(f"[existe]  {rel(destino)}")
        return destino

    # O nome definitivo só aparece quando o download termina
    parcial = destino.parent / (destino.name + ".part")
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resposta, open(parcial, "wb") as saida:
        while True:
            pedaco = resposta.read(1 << 20)
            if not pedaco:
                break
            saida.write(pedaco)
    parcial.replace(destino)
    print(f"[baixado] {rel(destino)}  ({destino.stat().st_size / 1e6:.1f} MB)")
    return destino


def ler_checklist() -> pd.DataFrame:
    if not ARQ_CHECKLIST.exists():
        return pd.DataFrame(columns=COLUNAS_CHECKLIST)
    return pd.read_csv(ARQ_CHECKLIST, sep="\t", dtype=str, keep_default_na=False)


def registrar(caminho: Path, url: str = "", fonte: str = "", checksum: bool = True) -> None:
    """Inclui ou atualiza a linha do arquivo no checklist em disco."""
    caminho = Path(caminho)
    if caminho.exists():
        linha = {"arquivo": rel(caminho), "status": "ok",
                 "bytes": str(caminho.stat().st_size),
                 "sha256": sha256(caminho) if checksum else "", "url": url, "fonte": fonte}
    else:
        linha = {"arquivo": rel(caminho), "status": "AUSENTE", "bytes": "0",
                 "sha256": "", "url": url, "fonte": fonte}

    checklist = ler_checklist()
    checklist = checklist[checklist["arquivo"] != linha["arquivo"]]
    checklist = pd.concat([checklist, pd.DataFrame([linha])], ignore_index=True)
    ARQ_CHECKLIST.parent.mkdir(parents=True, exist_ok=True)
    checklist.sort_values("arquivo").to_csv(ARQ_CHECKLIST, sep="\t", index=False)


def exigir_colunas(df: pd.DataFrame, colunas: list[str], nome: str) -> None:
    faltando = [c for c in colunas if c not in df.columns]
    if faltando:
        raise ValueError(f"{nome}: colunas obrigatórias ausentes {faltando}.\n"
                         f"Colunas presentes: {list(df.columns)}")


def simbolo_do_alvo(alvo) -> str:
    """'ELF1-human' -> 'ELF1'; 'YY1 (Homo sapiens)' -> 'YY1'."""
    if pd.isna(alvo):
        return ""
    s = re.sub(r"-human$", "", str(alvo).strip(), flags=re.IGNORECASE)
    s = re.sub(r"\s*\(.*?\)\s*", "", s).strip()
    return re.split(r"[:\s/]+", s)[0].strip().upper()


def normalizar(tfs) -> set[str]:
    return {str(t).strip().upper() for t in tfs if str(t).strip()}


def gravar_grupos(grupos_tfs: dict[str, list[str]], fonte: str,
                  criterio_manual: bool = False) -> pd.DataFrame:
    """Grava a composição dos grupos em outs/grupos_tfs.csv, com o resumo por grupo e as
    sobreposições, e devolve o resumo. Usada pela consolidação automática (3.8) e pela lista
    manual (3.9)."""
    def criterio(chave: str) -> str:
        c = GRUPOS[chave].criterio
        return f"lista manual; critério de referência: {c}" if criterio_manual else c

    tabela = pd.DataFrame(
        [{"grupo": chave, "rotulo": GRUPOS[chave].rotulo,
          "criterio": criterio(chave), "tf": tf}
         for chave, tfs in grupos_tfs.items() for tf in tfs],
        columns=["grupo", "rotulo", "criterio", "tf"],
    )
    ARQ_GRUPOS.parent.mkdir(parents=True, exist_ok=True)
    tabela.to_csv(ARQ_GRUPOS, index=False)

    resumo = (tabela.groupby(["grupo", "rotulo", "criterio"], as_index=False)
              .size().rename(columns={"size": "n_tfs"}))
    for chave in grupos_tfs:
        if chave not in set(resumo["grupo"]):
            resumo.loc[len(resumo)] = [chave, GRUPOS[chave].rotulo, criterio(chave), 0]
    resumo.to_csv(ARQ_GRUPOS_RESUMO, index=False)

    sobreposicoes = []
    chaves = list(grupos_tfs)
    for i, a in enumerate(chaves):
        for b in chaves[i + 1:]:
            comum = sorted(set(grupos_tfs[a]) & set(grupos_tfs[b]))
            if comum:
                sobreposicoes.append({"grupo_a": a, "grupo_b": b,
                                      "n": len(comum), "tfs": ", ".join(comum)})
    pd.DataFrame(sobreposicoes, columns=["grupo_a", "grupo_b", "n", "tfs"]).to_csv(
        ARQ_GRUPOS_SOBREPOSICOES, index=False)
    registrar(ARQ_GRUPOS, fonte=fonte, checksum=False)

    print(f"Pares (grupo, fator): {len(tabela)}; fatores distintos: {tabela['tf'].nunique()}")
    for s in sobreposicoes:
        print(f"  sobreposição {s['grupo_a']} x {s['grupo_b']}: {s['tfs']}")
    return resumo

## 0.6. Registro do ambiente

Data da execução, sistema operacional e versões do Python e dos pacotes, gravados em
`outs/ambiente_aquisicao_dados.json`. Quando um resultado muda de uma execução para outra, esse
registro ajuda a separar mudança nos dados de mudança de versão das bibliotecas. `pyscenic` e
`ctxcore` aparecem como `"ausente"` quando o notebook roda fora do ambiente do pipeline.

In [ ]:
def versao_pacote(nome: str) -> str:
    try:
        return version(nome)
    except PackageNotFoundError:
        return "ausente"


AMBIENTE = {
    "data_execucao_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "plataforma": platform.platform(),
    "raiz_projeto": str(ROOT),
    "pacotes": {p: versao_pacote(p) for p in
                ["pandas", "numpy", "pyarrow", "openpyxl", "pyscenic", "ctxcore"]},
}

ARQ_AMBIENTE.write_text(json.dumps(AMBIENTE, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(AMBIENTE, indent=2, ensure_ascii=False))

# 1. Aquisição dos dados públicos

A matriz de expressão, os metadados do ENCODE e os bancos de referência do cisTarget são
baixados nesta seção. As duas tabelas suplementares definidas na Seção 0.2 precisam ser obtidas
à mão nas publicações e colocadas em `downloads/literatura/`. Sem elas a Seção 3 não roda, e o
checklist da Seção 1.4 marca como `AUSENTE` a que estiver faltando.

## 1.1. Matriz de expressão de célula única

Amostra `GSM5677000` do GEO: registro em texto e arquivos suplementares. Se a listagem do
diretório do GEO falhar, a célula avisa, e os arquivos podem ser baixados à mão para
`downloads/geo/GSM5677000/`.

In [ ]:
URL_GEO_TEXTO = (f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
                 f"?acc={GSM_SCRNA}&targ=self&form=text&view=quick")
URL_GEO_SUPPL = (f"https://ftp.ncbi.nlm.nih.gov/geo/samples/"
                 f"{GSM_SCRNA[:-3]}nnn/{GSM_SCRNA}/suppl/")


def listar_diretorio_ftp(url_dir: str) -> list[str]:
    req = urllib.request.Request(url_dir, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=120) as r:
        html = r.read().decode("utf-8", errors="ignore")
    nomes = re.findall(r'href="([^"?#/][^"]*)"', html)
    return sorted({n for n in nomes if not n.lower().startswith("http")})


baixar(URL_GEO_TEXTO, DIR_GEO / f"{GSM_SCRNA}_registro.txt")

try:
    arquivos_suppl = listar_diretorio_ftp(URL_GEO_SUPPL)
    print(f"Arquivos suplementares em {GSM_SCRNA}: {arquivos_suppl}")
    for nome in arquivos_suppl:
        baixar(URL_GEO_SUPPL + nome, DIR_GEO / nome)
except Exception as erro:
    print(f"[ATENÇÃO] Listagem de {URL_GEO_SUPPL} indisponível ({erro}).\n"
          f"          Baixe os arquivos da página do {GSM_SCRNA} para {rel(DIR_GEO)}.")

for arq in sorted(DIR_GEO.glob("*")):
    registrar(arq, url=URL_GEO_SUPPL, fonte=f"GEO {GSM_SCRNA}")

## 1.2. Metadados do ENCODE

Ensaios de ChIP-seq de fator de transcrição em HepG2 com status liberado. O portal devolve um
TSV com uma linha por arquivo. Esse conteúdo muda à medida que experimentos são
liberados, reprocessados ou arquivados, então uma consulta nova não reproduz necessariamente a
original; o SHA-256 gravado no checklist (Seção 1.4) identifica a versão usada.

In [ ]:
URL_ENCODE_METADATA = (
    "https://www.encodeproject.org/metadata/"
    "?replicates.library.biosample.donor.organism.scientific_name=Homo+sapiens"
    "&assay_title=TF+ChIP-seq"
    "&status=released"
    "&biosample_ontology.term_name=HepG2"
    "&type=Experiment"
)

baixar(URL_ENCODE_METADATA, ENCODE_METADATA_TSV)
registrar(ENCODE_METADATA_TSV, url=URL_ENCODE_METADATA, fonte="ENCODE portal")

metadata = pd.read_csv(ENCODE_METADATA_TSV, sep="\t", low_memory=False)
print(f"{len(metadata)} arquivos de {metadata['Experiment accession'].nunique()} experimentos")

## 1.3. Bancos de referência do cisTarget

Arquivos padrão do laboratório Aerts para hg38, que formam a condição de referência, só com
motivos: a lista de fatores de transcrição, a anotação motivo-gene (`.tbl`) e o banco de
rankings, um feather de cerca de 1,2 GB.

In [ ]:
URL_ALLTFS = "https://raw.githubusercontent.com/aertslab/SCENICprotocol/master/example/allTFs_hg38.txt"
URL_TBL_MOTIFS = "https://resources.aertslab.org/cistarget/motif2tf/motifs-v9-nr.hgnc-m0.001-o0.0.tbl"
URL_FEATHER_MOTIFS = (
    "https://resources.aertslab.org/cistarget/databases/homo_sapiens/hg38/refseq_r80/"
    "mc9nr/gene_based/hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather"
)

for caminho, url in [(ALLTFS, URL_ALLTFS),
                     (TBL_MOTIFS, URL_TBL_MOTIFS),
                     (FEATHER_MOTIFS, URL_FEATHER_MOTIFS)]:
    baixar(url, caminho)
    registrar(caminho, url=url, fonte="resources.aertslab.org")

## 1.4. Checklist dos dados de entrada

Caminho, tamanho de cada arquivo de entrada, em `outs/checklist_dados_entrada.tsv`.
O hash permite verificar se o arquivo usado hoje é o mesmo de uma execução anterior, e o par
URL e hash identifica arquivos grandes demais para o material suplementar, como o feather de
motivos. Arquivo esperado que não está no disco aparece com status `AUSENTE`.

Cada chamada de `registrar` atualiza o arquivo em disco, por isso o checklist fica completo
mesmo quando só uma seção é executada.

In [ ]:
for caminho in [ARQ_PENG_FIG2B, ARQ_TFS_EXPRESSOS]:
    registrar(caminho, fonte="tabela suplementar")

checklist = ler_checklist()
print(f"{len(checklist)} arquivos ({(checklist['status'] == 'AUSENTE').sum()} ausentes)")
checklist[["arquivo", "status", "bytes"]]

# 2. Seleção do melhor experimento de ChIP-seq por fator

Parte dos fatores tem mais de um experimento no ENCODE. A escolha segue os critérios de
Moyers et al., aplicados em ordem até sobrar um candidato por fator:

1. escopo fixo: HepG2, ChIP-seq de fator de transcrição, arquivo liberado, GRCh38 e bigWig de
   `signal p-value`;
2. replicatas biológicas `{1, 2}`; na falta delas, `{1}`; sem nenhuma das duas, o fator é
   descartado;
3. menor número de auditorias inaceitáveis, sem descartar o fator quando todos os seus
   experimentos têm alguma;
4. no empate, prefere-se o experimento sem a auditoria mais grave da lista de Moyers et al.;
   persistindo o empate, sem a segunda mais grave, e assim por diante;
5. maior número de picos, estimado pelo tamanho do arquivo de picos (`optimal IDR thresholded
   peaks`) do mesmo experimento;
6. por último, pipeline do ENCODE mais recente, liberação mais recente e arquivo maior.

O resultado fica em `analise/best_bigwig_per_tf3.tsv`, que é o que as seções seguintes leem.

In [ ]:
ESCOPO = {
    "Biosample term name": "HepG2",
    "Assay": "TF ChIP-seq",
    "File Status": "released",
    "File assembly": "GRCh38",
    "File format": "bigWig",
    "Output type": "signal p-value",
}

# Ordem de severidade conforme Moyers et al.
AUDITORIAS_INACEITAVEIS = [
    "failed idr",
    "low read depth",
    "control low read depth",
    "extremely low read depth",
    "poor library analysis",
    "severe bottlenecking",
]

COLUNAS_SAIDA = [
    "Experiment target", "Experiment accession", "File accession", "File download URL",
    "File assembly", "Output type", "Biological replicate(s)", "n_unacceptable",
    "Audit ERROR", "Audit NOT_COMPLIANT", "Audit WARNING",
    "peaks_file_accession", "peaks_file_url", "peaks_file_size",
    "File analysis title", "Experiment date released",
]


def conjunto_replicatas(valor) -> set[int]:
    """'1, 2' -> {1, 2}."""
    if pd.isna(valor):
        return set()
    return {int(n) for n in re.findall(r"\d+", str(valor))}


def texto_auditorias(linha: pd.Series) -> str:
    partes = [str(linha[c]) for c in ["Audit ERROR", "Audit NOT_COMPLIANT", "Audit WARNING"]
              if c in linha and not pd.isna(linha[c])]
    return " | ".join(partes).lower()


def vetor_severidade(linha: pd.Series) -> tuple[int, ...]:
    t = texto_auditorias(linha)
    return tuple(int(frase in t) for frase in AUDITORIAS_INACEITAVEIS)


def versao_pipeline(titulo_analise) -> int:
    """'ENCODE4 ...' -> 4."""
    if pd.isna(titulo_analise):
        return -1
    m = re.search(r"encode(\d+)", str(titulo_analise).lower())
    return int(m.group(1)) if m else -1

In [ ]:
def melhor_bigwig_por_tf(metadata: pd.DataFrame) -> pd.DataFrame:
    exigir_colunas(metadata, list(ESCOPO) + [
        "Biological replicate(s)", "Experiment accession", "File accession",
        "File analysis title", "Experiment date released", "Size",
    ], "metadata do ENCODE")

    df = metadata.copy()
    df["Experiment date released"] = pd.to_datetime(df["Experiment date released"], errors="coerce")
    df["Size"] = pd.to_numeric(df["Size"], errors="coerce")

    mascara_escopo = pd.Series(True, index=df.index)
    for coluna, valor in ESCOPO.items():
        mascara_escopo &= df[coluna] == valor
    cand = df[mascara_escopo].copy()
    if cand.empty:
        raise ValueError("Nenhum arquivo passou no filtro de escopo.")

    # rank 0 = replicatas {1,2}, 1 = {1}, 9 = qualquer outra combinação
    rep = cand["Biological replicate(s)"].apply(conjunto_replicatas)
    cand["rank_replicata"] = rep.map(lambda s: 0 if s == {1, 2} else (1 if s == {1} else 9))
    melhor_rank = cand.groupby("Experiment target")["rank_replicata"].transform("min")
    cand = cand[(cand["rank_replicata"] == melhor_rank) & (cand["rank_replicata"] < 9)].copy()

    # Proxy de número de picos: um arquivo de picos representativo por experimento
    picos = df[
        (df["Biosample term name"] == ESCOPO["Biosample term name"])
        & (df["Assay"] == ESCOPO["Assay"])
        & (df["File Status"] == ESCOPO["File Status"])
        & (df["File assembly"] == ESCOPO["File assembly"])
        & (df["Output type"].astype(str).str.lower() == "optimal idr thresholded peaks")
    ].copy()
    picos = picos[picos["Biological replicate(s)"].apply(conjunto_replicatas) == {1, 2}].copy()
    picos["rank_formato"] = (picos["File format"]
                             .map({"bed narrowPeak": 0, "bigBed narrowPeak": 1})
                             .fillna(9).astype(int))
    picos_melhor = (
        picos.sort_values(["Experiment accession", "rank_formato", "Size"],
                          ascending=[True, True, False], kind="mergesort")
        .groupby("Experiment accession", as_index=False).head(1)
        .rename(columns={"File accession": "peaks_file_accession",
                         "File download URL": "peaks_file_url",
                         "Size": "peaks_file_size"})
        [["Experiment accession", "peaks_file_accession", "peaks_file_url", "peaks_file_size"]]
    )
    cand = cand.merge(picos_melhor, on="Experiment accession", how="left")

    severidade = cand.apply(vetor_severidade, axis=1)
    cand["n_unacceptable"] = severidade.map(sum)
    for i, nome in enumerate(AUDITORIAS_INACEITAVEIS):
        cand[f"sev_{i}"] = severidade.map(lambda v, i=i: v[i])

    # Fica o menor n_unacceptable de cada fator, mesmo que não seja zero
    minimo = cand.groupby("Experiment target")["n_unacceptable"].transform("min")
    cand = cand[cand["n_unacceptable"] == minimo].copy()

    cand["rank_pipeline"] = cand["File analysis title"].apply(versao_pipeline)
    colunas_ordem = (["Experiment target", "n_unacceptable"]
                     + [f"sev_{i}" for i in range(len(AUDITORIAS_INACEITAVEIS))]
                     + ["peaks_file_size", "rank_pipeline", "Experiment date released", "Size"])
    crescente = [True, True] + [True] * len(AUDITORIAS_INACEITAVEIS) + [False] * 4

    melhor = (cand.sort_values(colunas_ordem, ascending=crescente, kind="mergesort")
              .groupby("Experiment target", as_index=False).head(1))

    return melhor[[c for c in COLUNAS_SAIDA if c in melhor.columns]].reset_index(drop=True)

In [ ]:
def gerar_encode_melhor() -> None:
    metadata = pd.read_csv(ENCODE_METADATA_TSV, sep="\t", low_memory=False)
    melhor = melhor_bigwig_por_tf(metadata)
    ENCODE_BEST_TSV.parent.mkdir(parents=True, exist_ok=True)
    melhor.to_csv(ENCODE_BEST_TSV, sep="\t", index=False)
    print(f"Linhas no metadata bruto: {len(metadata)}")


encode_melhor = carregar_ou_gerar(
    ENCODE_BEST_TSV,
    gerar_encode_melhor,
    lambda: pd.read_csv(ENCODE_BEST_TSV, sep="\t", low_memory=False),
)
registrar(ENCODE_BEST_TSV, fonte="derivado do metadata do ENCODE")

print(f"Fatores com experimento selecionado: {len(encode_melhor)}")
print(f"Sem auditoria inaceitável: {(encode_melhor['n_unacceptable'] == 0).sum()}")
encode_melhor.head()

# 3. Definição dos seis grupos de fatores de transcrição

Cada grupo parte de uma lista primária, reduzida aos fatores testáveis: expressos em HepG2 e com
ChIP-seq selecionado no ENCODE na Seção 2. Nem todos os grupos passam pelos dois filtros. Os
centrais e os pioneiros preditos passam só pelo do ENCODE; no caso dos preditos, porque a fonte
já exigiu expressão na linhagem (Seção 3.6).

As listas curtas estão escritas no próprio notebook, nas Seções 3.1, 3.4, 3.5 e 3.7. As duas longas
vêm das tabelas suplementares: os fatores expressos em HepG2, de Moyers et al., e o ranking de
enrichment score da Figura 2b de Peng et al., de onde saem os canônicos preditos. Ficam fora do
notebook porque têm centenas de linhas e acompanham o trabalho como material suplementar.

A composição final dos grupos é gravada em `outs/grupos_tfs.csv` na Seção 3.8. A Seção 3.9,
opcional, permite escrever esse arquivo à mão, sem rodar as anteriores.

## 3.1. Listas primárias

In [ ]:
# Odom et al., Tabela Suplementar 1
TFS_CENTRAIS_LITERATURA = ["CREB1", "FOXA2", "HNF1A", "HNF4A", "ONECUT1", "USF1"]

# Saint-André et al., Tabela S3, contexto HepG2
TFS_CENTRAIS_SE = ["EHF", "ELF1", "FOXQ1", "NANOG", "RREB1", "TBX2", "YY1"]

# Fatores pioneiros descritos na literatura
TFS_PIONEIROS_CONHECIDOS = [
    "FOXA1", "FOXA2", "FOXA3", "GATA1", "GATA2", "GATA3", "GATA4", "CEBPA", "CEBPB",
    "NFYA", "NFYB", "NFYC", "ESRRB", "POU5F1", "KLF4", "NEUROD1", "TP53", "FOS",
    "FOSL1", "FOSL2", "MAFG", "MAFF", "MAFK", "JUN", "JUNB", "JUND", "ATF2", "ATF3",
    "ATF4", "ATF6", "ATF7", "SPI1",
]

# Peng et al., Supplementary file 1, Tabela 4, linhagem HepG2
TFS_PIONEIROS_PREDITOS_TAB4 = [
    "BCL6", "CEBPA", "CEBPB", "CEBPD", "CEBPG", "ESRRA", "FOSL2", "FOXA1", "FOXA2",
    "FOXA3", "GATA4", "HLF", "HNF4A", "MAFF", "MAFG", "MEF2A", "NFIL3", "NR2F1",
    "NR5A1", "PBX2", "PPARG", "RXRB", "TEAD1",
]

## 3.2. Conjuntos usados nos filtros

Fatores expressos em HepG2, fatores com ChIP-seq selecionado na Seção 2 e as duas listas de
pioneiros, em forma de conjunto. O de pioneiros preditos é usado também na 3.7, para que nenhum
deles apareça entre os canônicos preditos.

In [ ]:
encode_melhor = pd.read_csv(ENCODE_BEST_TSV, sep="\t", low_memory=False)
tfs_expressos = pd.read_csv(ARQ_TFS_EXPRESSOS)
exigir_colunas(tfs_expressos, ["Factor"], rel(ARQ_TFS_EXPRESSOS))

SET_EXPRESSOS = normalizar(tfs_expressos["Factor"].unique())
SET_ENCODE = normalizar(encode_melhor["Experiment target"].map(simbolo_do_alvo))
SET_PIONEIROS_CONHECIDOS = normalizar(TFS_PIONEIROS_CONHECIDOS)
SET_PIONEIROS_PREDITOS = normalizar(TFS_PIONEIROS_PREDITOS_TAB4)

print(f"Expressos em HepG2:             {len(SET_EXPRESSOS)}")
print(f"Com ChIP-seq no ENCODE:         {len(SET_ENCODE)}")
print(f"Expressos e com ChIP-seq:       {len(SET_EXPRESSOS & SET_ENCODE)}")
print(f"Pioneiros descritos:            {len(SET_PIONEIROS_CONHECIDOS)}")
print(f"Pioneiros preditos (Tabela 4):  {len(SET_PIONEIROS_PREDITOS)}")

grupos_tfs: dict[str, list[str]] = {}

## 3.3. Centrais de literatura e centrais de super-enhancer

Listas de Odom et al. e de Saint-André et al., restritas aos fatores com ChIP-seq no ENCODE.

In [ ]:
for chave, lista in [("centrais_literatura", TFS_CENTRAIS_LITERATURA),
                     ("centrais_se", TFS_CENTRAIS_SE)]:
    primaria = normalizar(lista)
    grupos_tfs[chave] = sorted(primaria & SET_ENCODE)
    print(f"{GRUPOS[chave].rotulo}: {len(grupos_tfs[chave])} de {len(primaria)}")
    print(f"  {grupos_tfs[chave]}")
    fora = sorted(primaria - SET_ENCODE)
    if fora:
        print(f"  fora do ENCODE: {fora}")

## 3.4. Pioneiros testáveis de literatura

Pioneiros descritos que estão expressos em HepG2 e têm ChIP-seq no ENCODE, menos os listados em
`TFs_sem_evidencia`. Essa lista é editável: fator incluído nela sai do grupo final, e uma lista
vazia desliga o filtro. A saída mostra em qual filtro caiu cada fator excluído.

Os fatores removidos por falta de evidência continuam em `SET_PIONEIROS_CONHECIDOS`, então
continuam fora do ranking de canônicos preditos da Seção 3.7. Saem do grupo de pioneiros, mas
não viram candidatos a canônicos.

In [ ]:
# Pioneiros descritos que ficam fora do grupo final por falta de evidência. Editável;
# lista vazia desliga o filtro.
TFs_sem_evidencia = [
    "ATF2", "ATF3", "FOSL1", "FOSL2", "FOXA2", "JUND", "MAFF", "MAFK", "NFYA",
]
SET_SEM_EVIDENCIA = normalizar(TFs_sem_evidencia)

_p = SET_PIONEIROS_CONHECIDOS
_testaveis = _p & SET_EXPRESSOS & SET_ENCODE
grupos_tfs["pioneiros_literatura"] = sorted(_testaveis - SET_SEM_EVIDENCIA)

print(f"{GRUPOS['pioneiros_literatura'].rotulo}: "
      f"{len(grupos_tfs['pioneiros_literatura'])} de {len(_p)}")
print(f"  {grupos_tfs['pioneiros_literatura']}")
print(f"  excluídos por expressão:          {sorted((_p & SET_ENCODE) - SET_EXPRESSOS)}")
print(f"  excluídos por ENCODE:             {sorted((_p & SET_EXPRESSOS) - SET_ENCODE)}")
print(f"  excluídos pelos dois:             {sorted(_p - SET_EXPRESSOS - SET_ENCODE)}")
print(f"  excluídos por falta de evidência: {sorted(_testaveis & SET_SEM_EVIDENCIA)}")

# Nomes da lista que não teriam efeito, para pegar erro de digitação
_ja_fora = sorted((SET_SEM_EVIDENCIA & _p) - _testaveis)
if _ja_fora:
    print(f"  [aviso] em TFs_sem_evidencia, mas já excluídos por expressão ou ENCODE: {_ja_fora}")
_nao_pioneiros = sorted(SET_SEM_EVIDENCIA - _p)
if _nao_pioneiros:
    print(f"  [aviso] em TFs_sem_evidencia, mas fora de TFS_PIONEIROS_CONHECIDOS: {_nao_pioneiros}")

## 3.5. Canônicos testáveis de literatura


In [ ]:
TFS_CANONICOS_LITERATURA = ["GATAD1", "MIXL1", "SMAD3", "TFDP2", "ZBTB25"]

SET_CANONICOS_LITERATURA = normalizar(TFS_CANONICOS_LITERATURA)
grupos_tfs["canonicos_literatura"] = sorted(
    SET_CANONICOS_LITERATURA & SET_EXPRESSOS & SET_ENCODE)

print(f"{GRUPOS['canonicos_literatura'].rotulo}: "
      f"{len(grupos_tfs['canonicos_literatura'])} de {len(SET_CANONICOS_LITERATURA)}")
print(f"  {grupos_tfs['canonicos_literatura']}")
_fora_cl = sorted(SET_CANONICOS_LITERATURA - (SET_EXPRESSOS & SET_ENCODE))
if _fora_cl:
    print(f"  excluídos por expressão ou ENCODE: {_fora_cl}")

## 3.6. Pioneiros testáveis preditos

Fatores da Tabela 4 do Supplementary file 1 de Peng et al., restrita a HepG2, sem os pioneiros
já conhecidos, para que fiquem só as predições novas. O filtro de expressão não é reaplicado
porque a própria tabela só inclui fatores com RPKM maior ou igual a 10 na linhagem, segundo
dados de expressão do Roadmap Epigenomics.

In [ ]:
_preditos_novos = SET_PIONEIROS_PREDITOS - SET_PIONEIROS_CONHECIDOS
grupos_tfs["pioneiros_preditos"] = sorted(_preditos_novos & SET_ENCODE)

print(f"{GRUPOS['pioneiros_preditos'].rotulo}: "
      f"{len(grupos_tfs['pioneiros_preditos'])} de {len(SET_PIONEIROS_PREDITOS)}")
print(f"  {grupos_tfs['pioneiros_preditos']}")
print(f"  removidos por serem pioneiros conhecidos: "
      f"{sorted(SET_PIONEIROS_PREDITOS & SET_PIONEIROS_CONHECIDOS)}")
print(f"  removidos por ausência no ENCODE:         {sorted(_preditos_novos - SET_ENCODE)}")

## 3.7. Canônicos testáveis preditos


In [ ]:
QUANTIL_CANONICOS = 0.25

# Tirados de outs/canonicos_preditos_ranking.csv da posição 1 em diante, até o tamanho
# do grupo de pioneiros preditos
TFS_CANONICOS_PREDITOS = [
    "E2F1", "E2F2", "E2F4", "ELF4", "ELK1", "ELK4", "ETS1",
    "ETV6", "GMEB2", "SP1", "SP2", "TFDP1", "ZBTB14", "ZBTB33",
]

fig2b = pd.read_excel(ARQ_PENG_FIG2B, sheet_name="Figure2b_Enrichment_score")
exigir_colunas(fig2b, ["cell_line", "TF_name", "Enrichment score"], rel(ARQ_PENG_FIG2B))

ranking = (fig2b[fig2b["cell_line"] == "HepG2"]
           .sort_values("Enrichment score", ascending=True)
           .reset_index(drop=True))
ranking["TF_norm"] = ranking["TF_name"].map(lambda s: str(s).strip().upper())

# O quartil é calculado sobre o ranking completo, antes dos filtros
n_quartil = math.ceil(len(ranking) * QUANTIL_CANONICOS)
quartil = ranking.head(n_quartil)

# Fator incluído em qualquer um destes grupos sai do ranking elegível
GRUPOS_EXCLUDENTES = {
    "pioneiros de literatura": SET_PIONEIROS_CONHECIDOS,
    "canônicos de literatura": SET_CANONICOS_LITERATURA,
    "pioneiros preditos":      SET_PIONEIROS_PREDITOS,
}
JA_CLASSIFICADOS = set().union(*GRUPOS_EXCLUDENTES.values())

elegiveis = (quartil[quartil["TF_norm"].isin(SET_EXPRESSOS)
                     & quartil["TF_norm"].isin(SET_ENCODE)
                     & ~quartil["TF_norm"].isin(JA_CLASSIFICADOS)]
             .reset_index(drop=True))
elegiveis.insert(0, "posicao", elegiveis.index + 1)

colunas = [c for c in ["posicao", "TF_norm", "Enrichment score", "p_value"]
           if c in elegiveis.columns]
elegiveis[colunas].to_csv(ARQ_CANONICOS_RANKING, index=False)
registrar(ARQ_CANONICOS_RANKING, fonte="derivado da Figura 2b de Peng et al.", checksum=False)

print(f"Quartil inferior ({QUANTIL_CANONICOS:.0%}): {n_quartil} de {len(ranking)} fatores, "
      f"enrichment score até {quartil['Enrichment score'].max():.4f}")
print(f"  removidos por expressão ou ENCODE: "
      f"{sorted(set(quartil['TF_norm']) - (SET_EXPRESSOS & SET_ENCODE))}")
for _rotulo, _conjunto in GRUPOS_EXCLUDENTES.items():
    print(f"  removidos por já serem {_rotulo}: {sorted(set(quartil['TF_norm']) & _conjunto)}")
print(f"Elegíveis no quartil: {len(elegiveis)}, gravados em {rel(ARQ_CANONICOS_RANKING)}")

curados = normalizar(TFS_CANONICOS_PREDITOS)
fora = sorted(curados - set(elegiveis["TF_norm"]))
if fora:
    raise ValueError(f"Fatores de TFS_CANONICOS_PREDITOS ausentes do ranking elegível: "
                     f"{fora}. Refaça a seleção a partir de "
                     f"{rel(ARQ_CANONICOS_RANKING)} e atualize a lista.")

grupos_tfs["canonicos_preditos"] = sorted(curados)
posicoes = elegiveis.loc[elegiveis["TF_norm"].isin(curados), "posicao"].tolist()
print(f"\n{GRUPOS['canonicos_preditos'].rotulo}: {len(grupos_tfs['canonicos_preditos'])}")
print(f"  {grupos_tfs['canonicos_preditos']}")
print(f"  posições no ranking elegível: {posicoes}")

## 3.8. Consolidação dos grupos

Grava a composição final em `outs/grupos_tfs.csv`, que é a entrada da Seção 4, além de um
resumo por grupo e da lista de fatores presentes em mais de um grupo.

Sobreposição com os grupos de centrais é esperada, já que um fator central também pode ser
pioneiro, como FOXA2 e HNF4A. Entre os quatro grupos de classificação, pioneiros e canônicos de
literatura e preditos, não deve haver nenhuma; se aparecer, há erro em alguma lista de origem.

In [ ]:
resumo = gravar_grupos(grupos_tfs, fonte="derivado das listas primárias")
resumo

## 3.9. Opcional: composição dos grupos escrita à mão

**Esta seção não faz parte do fluxo padrão e não é obrigatória.** Use-a só quando quiser definir
os grupos manualmente, sem rodar as Seções 1 a 3.8, por exemplo para testar outra composição ou
para reproduzir a análise sem as tabelas suplementares. Ela precisa apenas da Seção 0.

Com `USAR_LISTA_MANUAL = False`, que é o padrão, a célula não faz nada, e rodar o notebook
inteiro não sobrescreve o que a Seção 3.8 gravou. Com `True`, ela grava `outs/grupos_tfs.csv`,
o resumo e as sobreposições a partir das listas abaixo, no mesmo formato da 3.8, e a Seção 4 lê
esse arquivo normalmente. A coluna `criterio` passa a dizer que a lista é manual.

As listas vêm preenchidas com a composição atual dos grupos, como ponto de partida. Nenhum filtro
é aplicado: o que estiver escrito é o que entra. A célula só avisa quando um fator não tem
ChIP-seq selecionado no ENCODE, se a tabela da Seção 2 existir, e interrompe a execução se um
fator aparecer em dois dos quatro grupos de classificação.

Se `outs/fatores_a_processar.csv` já existir, apague-o antes de rodar a Seção 4, senão ela
continua usando a composição anterior.

In [ ]:
# OPCIONAL. Só use esta célula para definir os grupos à mão (ver o texto acima).
# Com USAR_LISTA_MANUAL = False ela não altera nada.
USAR_LISTA_MANUAL = False

GRUPOS_MANUAIS = {
    "centrais_literatura": [
        "CREB1", "FOXA2", "HNF1A", "HNF4A", "ONECUT1", "USF1",
    ],
    "centrais_se": [
        "ELF1", "FOXQ1", "RREB1", "TBX2", "YY1",
    ],
    "pioneiros_literatura": [
        "ATF4", "ATF6", "ATF7", "CEBPA", "CEBPB", "FOXA1", "FOXA3",
        "GATA2", "GATA4", "JUN", "MAFG", "NFYB", "NFYC", "TP53",
    ],
    "canonicos_literatura": [
        "GATAD1", "MIXL1", "SMAD3", "TFDP2", "ZBTB25",
    ],
    "pioneiros_preditos": [
        "BCL6", "CEBPD", "CEBPG", "ESRRA", "HLF", "HNF4A", "MEF2A",
        "NFIL3", "NR2F1", "NR5A1", "PBX2", "PPARG", "RXRB", "TEAD1",
    ],
    "canonicos_preditos": [
        "E2F1", "E2F2", "E2F4", "ELF4", "ELK1", "ELK4", "ETS1",
        "ETV6", "GMEB2", "SP1", "SP2", "TFDP1", "ZBTB14", "ZBTB33",
    ],
}

GRUPOS_CLASSIFICACAO = ["pioneiros_literatura", "canonicos_literatura",
                        "pioneiros_preditos", "canonicos_preditos"]

if not USAR_LISTA_MANUAL:
    print("Lista manual desativada; outs/grupos_tfs.csv não foi alterado.")
else:
    _desconhecidos = sorted(set(GRUPOS_MANUAIS) - set(GRUPOS))
    if _desconhecidos:
        raise ValueError(f"Chaves de grupo inexistentes: {_desconhecidos}. "
                         f"Chaves válidas: {list(GRUPOS)}")

    # Grupo omitido do dicionário fica vazio
    grupos_manuais = {chave: sorted(normalizar(GRUPOS_MANUAIS.get(chave, [])))
                      for chave in GRUPOS}

    # Os quatro grupos de classificação não podem compartilhar fatores
    for i, a in enumerate(GRUPOS_CLASSIFICACAO):
        for b in GRUPOS_CLASSIFICACAO[i + 1:]:
            comum = sorted(set(grupos_manuais[a]) & set(grupos_manuais[b]))
            if comum:
                raise ValueError(f"{a} e {b} compartilham {comum}. "
                                 f"Um fator não pode estar em duas classes.")

    # Aviso, sem bloquear: fator sem ChIP-seq não gera banco na Seção 4
    if ENCODE_BEST_TSV.exists():
        _encode = pd.read_csv(ENCODE_BEST_TSV, sep="\t", low_memory=False)
        _set_encode = normalizar(_encode["Experiment target"].map(simbolo_do_alvo))
        for chave, tfs in grupos_manuais.items():
            _sem = sorted(set(tfs) - _set_encode)
            if _sem:
                print(f"[aviso] {chave}: sem ChIP-seq selecionado no ENCODE: {_sem}")
    else:
        print(f"[aviso] {rel(ENCODE_BEST_TSV)} ausente; disponibilidade no ENCODE não conferida.")

    resumo = gravar_grupos(grupos_manuais, fonte="lista manual (Seção 3.9)",
                           criterio_manual=True)

    if ARQ_FATORES.exists():
        print(f"\n[aviso] {rel(ARQ_FATORES)} já existe e reflete a composição anterior. "
              f"Apague-o antes de rodar a Seção 4.")
    display(resumo)

# 4. Bancos cisTarget customizados

Para cada fator selecionado são gerados três arquivos:

| Arquivo | Conteúdo |
|---|---|
| `only_chip/<prefixo>_<TF>.feather` | linhas do feather global que pertencem ao fator |
| `motifs_plus_chip/hg38__motifs_plus_<TF>...feather` | banco de motivos acrescido dessas linhas |
| `<prefixo>_<TF>.tbl` | anotação de referência acrescida dessas linhas |

A condição experimental compara, fator a fator, a rede inferida com o banco de referência e a
rede inferida com o banco acrescido do ChIP-seq daquele fator. Por isso cada fator tem seu
próprio par de feather e `.tbl`, em vez de um banco único com todos os fatores.

O banco combinado e o `.tbl` são gerados a partir do recorte gravado em disco. Se o recorte de
um fator for refeito, os outros dois arquivos daquele fator precisam ser apagados para serem
gerados de novo.

## 4.0. Entradas desta seção

Tabela do ENCODE da Seção 2 e composição dos grupos da Seção 3, lidas do disco.

In [ ]:
encode_melhor = pd.read_csv(ENCODE_BEST_TSV, sep="\t", low_memory=False)

tabela_grupos = pd.read_csv(ARQ_GRUPOS)
exigir_colunas(tabela_grupos, ["grupo", "tf"], rel(ARQ_GRUPOS))
grupos_tfs = {chave: sorted(tabela_grupos.loc[tabela_grupos["grupo"] == chave, "tf"]
                            .astype(str).str.upper().unique())
              for chave in GRUPOS}

print(f"Fatores no ENCODE selecionado: {len(encode_melhor)}")
for chave, tfs in grupos_tfs.items():
    print(f"  {chave:<24} {len(tfs)}")

## 4.1. Feather global de tracks

Banco de rankings com todos os tracks de ChIP-seq, disponível no material suplementar do
trabalho. Precisa estar em `data/feathers/all_tracks_ofc/`.

In [ ]:
if not FEATHER_TRACKS_GLOBAL.exists():
    raise FileNotFoundError(f"Feather global ausente: {rel(FEATHER_TRACKS_GLOBAL)}")

COLUNA_ID_TRACKS = "tracks"
COLUNA_ID_MOTIFS = "motifs"


def coluna_identificadora(caminho: Path, coluna: str) -> list[str]:
    """Lê apenas a coluna identificadora, sem carregar a matriz de rankings."""
    return pf.read_table(caminho, columns=[coluna]).column(0).to_pylist()


registrar(FEATHER_TRACKS_GLOBAL, fonte="material suplementar", checksum=False)
print(f"{rel(FEATHER_TRACKS_GLOBAL)} "
      f"({FEATHER_TRACKS_GLOBAL.stat().st_size / 1e6:.0f} MB)")

## 4.2. Correspondência entre fator e tracks

Os identificadores de linha do feather global têm a forma `00005__ENCFF773TUX`. O código `ENCFF`
liga cada linha ao fator, pela tabela curada da Seção 2, e o mapa resultante é gravado em
`outs/mapa_tf_tracks.tsv`.

In [ ]:
def extrair_encff(identificador) -> str | None:
    """'00010__ENCFF123ABC' -> 'ENCFF123ABC'."""
    m = re.search(r"(ENCFF[0-9A-Z]+)", str(identificador), flags=re.IGNORECASE)
    return m.group(1).upper() if m else None


_chave = encode_melhor["File accession"].astype(str).str.strip().str.upper()
ENCFF_PARA_TF = dict(zip(_chave, encode_melhor["Experiment target"].map(simbolo_do_alvo)))
METADADOS_POR_ENCFF = (encode_melhor.assign(_k=_chave)
                       .set_index("_k").to_dict(orient="index"))


def gerar_mapa_tracks() -> None:
    tracks_globais = coluna_identificadora(FEATHER_TRACKS_GLOBAL, COLUNA_ID_TRACKS)
    linhas = []
    for identificador in tracks_globais:
        encff = extrair_encff(identificador)
        tf = ENCFF_PARA_TF.get(encff) if encff else None
        if tf:
            linhas.append({"tf": tf, "track": str(identificador), "encff": encff})

    mapa = (pd.DataFrame(linhas, columns=["tf", "track", "encff"])
            .drop_duplicates(subset=["tf", "track"], keep="last")
            .sort_values(["tf", "track"]))
    mapa.to_csv(ARQ_MAPA_TRACKS, sep="\t", index=False)
    print(f"Tracks no feather global: {len(tracks_globais)}; "
          f"sem correspondência: {len(tracks_globais) - len(mapa)}")


mapa_tracks = carregar_ou_gerar(
    ARQ_MAPA_TRACKS, gerar_mapa_tracks,
    lambda: pd.read_csv(ARQ_MAPA_TRACKS, sep="\t", dtype=str, keep_default_na=False))

TF_PARA_TRACKS = mapa_tracks.groupby("tf")["track"].apply(list).to_dict()
print(f"Fatores com ao menos um track: {len(TF_PARA_TRACKS)}")
print(f"Tracks mapeados: {len(mapa_tracks)}")

## 4.3. Fatores a processar

Pares (grupo, fator) com o número de tracks de cada fator, gravados em
`outs/fatores_a_processar.csv` e relidos na Seção 5. Fator sem nenhum track no feather global não
é processado e aparece listado na saída.

Este arquivo não é refeito se já existe. Depois de mudar os grupos na Seção 3, é preciso apagá-lo
antes de rodar a Seção 4 de novo.

In [ ]:
def gerar_lista_fatores() -> None:
    linhas = [{"grupo": chave, "tf": tf,
               "n_tracks": len(TF_PARA_TRACKS.get(tf, [])),
               "processar": bool(TF_PARA_TRACKS.get(tf))}
              for chave, tfs in grupos_tfs.items() for tf in tfs]
    pd.DataFrame(linhas, columns=["grupo", "tf", "n_tracks", "processar"]).to_csv(
        ARQ_FATORES, index=False)


fatores = carregar_ou_gerar(ARQ_FATORES, gerar_lista_fatores,
                            lambda: pd.read_csv(ARQ_FATORES))

alvos = list(fatores.loc[fatores["processar"], ["grupo", "tf"]].itertuples(index=False, name=None))
sem_track = list(fatores.loc[~fatores["processar"], ["grupo", "tf"]].itertuples(index=False, name=None))

print(f"Fatores a processar: {len(alvos)}")
for chave in grupos_tfs:
    print(f"  {chave:<24} {sum(1 for g, _ in alvos if g == chave)}")

if sem_track:
    print(f"\nSem track no feather global ({len(sem_track)}):")
    for chave, tf in sem_track:
        print(f"  {chave}: {tf}")

## 4.4. Recorte do feather global por fator

O feather global só é lido quando falta algum recorte, e o mesmo vale para o banco de motivos
(4.5) e para o `.tbl` de referência (4.6). Com todos os arquivos já em disco, a Seção 4 roda sem
carregar nenhuma matriz de rankings.

In [ ]:
_CACHE: dict[str, pd.DataFrame] = {}


def tracks_globais_df() -> pd.DataFrame:
    if "tracks" not in _CACHE:
        print("Carregando o feather global de tracks...")
        _CACHE["tracks"] = pd.read_feather(FEATHER_TRACKS_GLOBAL)
        print(f"  {_CACHE['tracks'].shape[0]} tracks x {_CACHE['tracks'].shape[1] - 1} genes")
    return _CACHE["tracks"]


def gerar_feather_only_chip(tf: str, destino: Path, forcar: bool = False) -> Path:
    if destino.exists() and not (forcar or FORCAR):
        return destino

    identificadores = set(TF_PARA_TRACKS.get(tf, []))
    if not identificadores:
        raise ValueError(f"{tf}: nenhum track correspondente.")

    df = tracks_globais_df()
    recorte = (df[df[COLUNA_ID_TRACKS].isin(identificadores)].reset_index(drop=True))
    if recorte.empty:
        raise ValueError(f"{tf}: filtro resultou em zero linhas.")

    destino.parent.mkdir(parents=True, exist_ok=True)
    recorte.to_feather(destino)
    return destino

## 4.5. Banco de rankings com motivos e ChIP-seq

As linhas de ChIP-seq do fator são acrescentadas ao final do banco de motivos. Cada linha de um
banco cisTarget é um ranking de genes calculado independentemente das outras, então as linhas de
motivos ficam como estavam. A coluna identificadora dos tracks, `tracks`, passa a se chamar
`motifs`, porque o arquivo resultante é lido como banco de motivos.

In [ ]:
def motifs_df() -> pd.DataFrame:
    if "motifs" not in _CACHE:
        print("Carregando o banco de motivos...")
        df = pd.read_feather(FEATHER_MOTIFS)
        if COLUNA_ID_MOTIFS not in df.columns:
            raise ValueError(f"{rel(FEATHER_MOTIFS)}: coluna '{COLUNA_ID_MOTIFS}' ausente.")
        _CACHE["motifs"] = df
        print(f"  {df.shape[0]} motivos x {df.shape[1] - 1} genes")
    return _CACHE["motifs"]


def gerar_feather_motifs_plus_chip(feather_chip: Path, destino: Path,
                                   forcar: bool = False) -> Path:
    if destino.exists() and not (forcar or FORCAR):
        return destino

    df_chip = pd.read_feather(feather_chip)
    if COLUNA_ID_TRACKS not in df_chip.columns:
        raise ValueError(f"{rel(feather_chip)}: coluna '{COLUNA_ID_TRACKS}' ausente.")

    base = motifs_df()
    genes_motifs = [c for c in base.columns if c != COLUNA_ID_MOTIFS]
    genes_chip = [c for c in df_chip.columns if c != COLUNA_ID_TRACKS]
    if set(genes_motifs) != set(genes_chip):
        faltam = sorted(set(genes_motifs) - set(genes_chip))
        sobram = sorted(set(genes_chip) - set(genes_motifs))
        raise ValueError(f"Conjunto de genes divergente.\n"
                         f"  ausentes no ChIP ({len(faltam)}): {faltam[:10]}\n"
                         f"  extras no ChIP ({len(sobram)}): {sobram[:10]}")

    chip_alinhado = (df_chip[genes_motifs + [COLUNA_ID_TRACKS]]
                     .rename(columns={COLUNA_ID_TRACKS: COLUNA_ID_MOTIFS}))
    combinado = pd.concat([base[genes_motifs + [COLUNA_ID_MOTIFS]], chip_alinhado],
                          ignore_index=True)

    if not combinado[COLUNA_ID_MOTIFS].is_unique:
        dups = combinado.loc[combinado[COLUNA_ID_MOTIFS].duplicated(), COLUNA_ID_MOTIFS]
        raise ValueError(f"Identificadores duplicados: {sorted(set(dups))[:10]}")

    destino.parent.mkdir(parents=True, exist_ok=True)
    combinado.to_feather(destino)
    return destino

## 4.6. Anotação `.tbl` customizada

O `.tbl` liga cada linha do banco de rankings a um gene. Nas linhas de ChIP-seq, o gene é o
próprio fator alvo do experimento. Os identificadores são lidos do recorte gravado em 4.4, de
modo que o `.tbl` nunca descreve uma linha que não está no banco, e os demais campos vêm da
tabela curada da Seção 2, pelo código `ENCFF`.

`similar_motif_id` e `orthologous_gene_name` ficam vazios, o que caracteriza a linha como
anotação direta do fator, sem herança por similaridade de motivo ou por ortologia.
`motif_similarity_qvalue` recebe 0,0 e `orthologous_identity` recebe 1,0, valores que passam por
qualquer limiar de filtragem do pySCENIC.

In [ ]:
MOTIF_SIMILARITY_QVALUE = "0.0"
ORTHOLOGOUS_IDENTITY = "1.0"


def tbl_base() -> pd.DataFrame:
    if "tbl" not in _CACHE:
        # keep_default_na=False preserva células vazias como texto vazio.
        df = pd.read_csv(TBL_MOTIFS, sep="\t", dtype=str, keep_default_na=False)
        exigir_colunas(df, ["#motif_id", "gene_name"], rel(TBL_MOTIFS))
        _CACHE["tbl"] = df
        print(f"  {len(df)} linhas, {len(df.columns)} colunas no .tbl de referência")
    return _CACHE["tbl"]


def _texto(valor) -> str:
    return "" if valor is None or pd.isna(valor) else str(valor).strip()


def linhas_chipseq_para_tbl(identificadores: list[str], colunas: list[str]) -> pd.DataFrame:
    linhas = []
    for identificador in identificadores:
        identificador = str(identificador).strip()
        encff = extrair_encff(identificador)
        meta = METADADOS_POR_ENCFF.get(encff, {}) if encff else {}
        tf = ENCFF_PARA_TF.get(encff, "") if encff else ""

        contexto = [_texto(meta.get(c)) for c in
                    ["Output type", "File assembly", "Experiment accession"]]
        descricao = "; ".join(
            [f"ChIP-seq track {identificador}"]
            + [f"{rotulo}={valor}" for rotulo, valor in
               [("target", tf), ("file", encff or ""),
                ("exp", _texto(meta.get("Experiment accession"))),
                ("assembly", _texto(meta.get("File assembly")))]
               if valor])

        linha = dict.fromkeys(colunas, "")
        linha["#motif_id"] = identificador
        linha["gene_name"] = tf
        for coluna, valor in [
            ("motif_name", f"{tf}_ChIPseq" if tf else "ChIPseq_track"),
            ("motif_description", " | ".join(p for p in contexto if p) or "ChIP-seq track"),
            ("source_name", "ENCODE"),
            ("source_version", _texto(meta.get("File assembly"))),
            ("description", descricao),
            ("motif_similarity_qvalue", MOTIF_SIMILARITY_QVALUE),
            ("orthologous_identity", ORTHOLOGOUS_IDENTITY),
        ]:
            if coluna in linha:
                linha[coluna] = valor
        linhas.append(linha)

    return pd.DataFrame(linhas, columns=colunas)


def gerar_tbl(feather_chip: Path, destino: Path, tf: str, forcar: bool = False) -> Path:
    if destino.exists() and not (forcar or FORCAR):
        return destino

    identificadores = coluna_identificadora(feather_chip, COLUNA_ID_TRACKS)
    if not identificadores:
        raise ValueError(f"{tf}: {rel(feather_chip)} não tem nenhum track.")

    base = tbl_base()
    novas = linhas_chipseq_para_tbl(identificadores, list(base.columns))
    sem_gene = novas[novas["gene_name"].str.strip() == ""]
    if len(sem_gene):
        raise ValueError(f"{tf}: {len(sem_gene)} track(s) sem gene associado.")
    if set(novas["gene_name"].str.upper()) - {tf.upper()}:
        raise ValueError(f"{tf}: o recorte contém track de outro fator "
                         f"({sorted(set(novas['gene_name']))}).")

    combinado = (pd.concat([base, novas], ignore_index=True)
                 .drop_duplicates(subset=["#motif_id", "gene_name"], keep="first"))
    destino.parent.mkdir(parents=True, exist_ok=True)
    combinado.to_csv(destino, sep="\t", index=False, na_rep="")
    return destino

## 4.7. Geração dos três arquivos por fator

Para cada fator são gerados, nesta ordem, o recorte, o banco combinado e o `.tbl`. Arquivo que já
existe é mantido, e a falha em um fator não interrompe os demais. O resultado de cada fator vai
para `outs/geracao_bancos_cistarget.csv`, que a Seção 4.8 usa para saber o que conferir.

Por padrão todos os grupos são processados. Para gerar só alguns, coloque as chaves deles
(Seção 0.3) em `GRUPOS_A_GERAR`; o registro da geração e a conferência passam então a cobrir só
esses grupos.

In [ ]:
GRUPOS_A_GERAR = None   # None processa todos; ex.: ["canonicos_literatura"]

a_gerar = [(g, tf) for g, tf in alvos if not GRUPOS_A_GERAR or g in GRUPOS_A_GERAR]
registro_geracao = []

for i, (chave_grupo, tf) in enumerate(a_gerar, start=1):
    caminhos = caminhos_do_fator(chave_grupo, tf)
    linha = {"grupo": chave_grupo, "tf": tf,
             "n_tracks": len(TF_PARA_TRACKS.get(tf, [])), "status": "ok", "erro": ""}
    try:
        gerar_feather_only_chip(tf, caminhos["only_chip"])
        gerar_feather_motifs_plus_chip(caminhos["only_chip"], caminhos["motifs_plus_chip"])
        gerar_tbl(caminhos["only_chip"], caminhos["tbl"], tf)
    except Exception as erro:
        linha["status"] = "FALHOU"
        linha["erro"] = f"{type(erro).__name__}: {erro}"
    registro_geracao.append(linha)
    print(f"[{i:>3}/{len(a_gerar)}] {chave_grupo:<22} {tf:<10} {linha['status']}")

pd.DataFrame(registro_geracao,
             columns=["grupo", "tf", "n_tracks", "status", "erro"]).to_csv(
    ARQ_GERACAO, index=False)

df_geracao = pd.read_csv(ARQ_GERACAO, keep_default_na=False)
falhas = df_geracao[df_geracao["status"] != "ok"]
print(f"\nConcluídos: {len(df_geracao) - len(falhas)} de {len(df_geracao)}")
for _, f in falhas.iterrows():
    print(f"  {f['grupo']} / {f['tf']}: {f['erro']}")

## 4.8. Conferência dos arquivos gerados

Confere, lendo só as colunas de identificadores, se os três arquivos de cada fator batem entre
si: o recorte só contém tracks do fator; o banco combinado tem exatamente as linhas do banco de
motivos mais as do recorte, sem identificador repetido; e o `.tbl` tem uma linha por track, com
`gene_name` igual ao fator e os campos de similaridade e ortologia vazios. Os fatores a conferir
vêm de `outs/geracao_bancos_cistarget.csv`.

In [ ]:
def conferir_fator(chave_grupo: str, tf: str) -> dict:
    caminhos = caminhos_do_fator(chave_grupo, tf)
    r = {"grupo": chave_grupo, "tf": tf, "ok": True, "observacoes": []}

    ausentes = [nome for nome, caminho in caminhos.items() if not caminho.exists()]
    if ausentes:
        return {**r, "ok": False, "observacoes": "; ".join(f"{a} ausente" for a in ausentes)}

    tracks_chip = coluna_identificadora(caminhos["only_chip"], COLUNA_ID_TRACKS)
    esperados = set(TF_PARA_TRACKS.get(tf, []))
    fora = [t for t in tracks_chip if t not in esperados]
    if fora:
        r["ok"] = False
        r["observacoes"].append(f"tracks de outro fator no recorte: {fora[:3]}")

    ids_combinado = coluna_identificadora(caminhos["motifs_plus_chip"], COLUNA_ID_MOTIFS)
    esperado = len(IDS_MOTIFS_BASE) + len(tracks_chip)
    if len(ids_combinado) != esperado:
        r["ok"] = False
        r["observacoes"].append(f"{len(ids_combinado)} linhas no banco combinado, "
                                f"esperado {esperado}")
    if len(set(ids_combinado)) != len(ids_combinado):
        r["ok"] = False
        r["observacoes"].append("identificadores duplicados no banco combinado")
    if set(ids_combinado) != SET_IDS_MOTIFS_BASE | set(tracks_chip):
        r["ok"] = False
        r["observacoes"].append("banco combinado não tem exatamente os motivos mais o recorte")

    tbl = pd.read_csv(caminhos["tbl"], sep="\t", dtype=str, keep_default_na=False)
    novas = tbl[tbl["#motif_id"].isin(tracks_chip)]
    if len(novas) != len(tracks_chip):
        r["ok"] = False
        r["observacoes"].append(f"{len(novas)} linhas de ChIP no .tbl, "
                                f"esperado {len(tracks_chip)}")
    if set(novas["gene_name"].str.upper()) - {tf.upper()}:
        r["ok"] = False
        r["observacoes"].append("gene_name divergente")
    for coluna in ["similar_motif_id", "orthologous_gene_name"]:
        if coluna in novas.columns and novas[coluna].str.strip().ne("").any():
            r["ok"] = False
            r["observacoes"].append(f"{coluna} preenchido")

    r["observacoes"] = "; ".join(r["observacoes"])
    return r


IDS_MOTIFS_BASE = coluna_identificadora(FEATHER_MOTIFS, COLUNA_ID_MOTIFS)
SET_IDS_MOTIFS_BASE = set(IDS_MOTIFS_BASE)

df_geracao = pd.read_csv(ARQ_GERACAO, keep_default_na=False)
a_conferir = df_geracao.loc[df_geracao["status"] == "ok", ["grupo", "tf"]]

conferencia = pd.DataFrame([conferir_fator(g, t) for g, t in
                            a_conferir.itertuples(index=False, name=None)])
if len(conferencia):
    conferencia.to_csv(ARQ_CONFERENCIA, index=False)
    problemas = conferencia[~conferencia["ok"]]
    print(f"Conferidos: {len(conferencia)}; com problema: {len(problemas)}")
    for _, p in problemas.iterrows():
        print(f"  {p['grupo']} / {p['tf']}: {p['observacoes']}")
else:
    print("Nada a conferir.")

# 5. Relatório final

Inventário dos arquivos da Seção 4 presentes em disco, montado a partir de
`outs/fatores_a_processar.csv`, com o volume total gerado e a lista das saídas em `outs/`.

In [ ]:
fatores = pd.read_csv(ARQ_FATORES)
alvos = list(fatores.loc[fatores["processar"], ["grupo", "tf"]].itertuples(index=False, name=None))

inventario = []
for chave_grupo, tf in alvos:
    for nome, caminho in caminhos_do_fator(chave_grupo, tf).items():
        inventario.append({"grupo": chave_grupo, "tf": tf, "arquivo_tipo": nome,
                           "arquivo": rel(caminho), "existe": caminho.exists(),
                           "bytes": caminho.stat().st_size if caminho.exists() else 0})

df_inventario = pd.DataFrame(
    inventario, columns=["grupo", "tf", "arquivo_tipo", "arquivo", "existe", "bytes"])
df_inventario.to_csv(ARQ_INVENTARIO, sep="\t", index=False)

tabela_grupos = pd.read_csv(ARQ_GRUPOS)

print(f"Raiz do projeto: {ROOT}")
print(f"Grupos com fatores: {tabela_grupos['grupo'].nunique()} de {len(GRUPOS)}; "
      f"fatores distintos: {tabela_grupos['tf'].nunique()}")
print(f"Arquivos esperados: {len(df_inventario)}; presentes: "
      f"{int(df_inventario['existe'].sum())}")
print(f"Volume gerado: {df_inventario['bytes'].sum() / 1e9:.1f} GB")
print("\nSaídas em outs/:")
for arq in sorted(OUTS.rglob("*")):
    if arq.is_file():
        print(f"  {rel(arq)}")